# Exercise 4: Text Generation using LSTM

### Objective
In this assignment, you will implement a character-level text generation model using Long Short-Term Memory (LSTM) networks in PyTorch. The goal is to understand how LSTMs work for sequential data and how to train them effectively to generate new text based on an input sequence.

You will follow the steps below:
1. Load and preprocess a text dataset
2. Character-level encoding by constructing the vocabulary and dictionary (2 points)
3. Batch generation for training (6 points)
4. Defining the character-level LSTM model (6 points)
5. Training loop (3 points)
6. Text generation using the trained model (3 points)


**IMPORTANT NOTE**: 
1. Remove the line `raise NotImplementedError()` from every cell where you write your code.
2. `skip_training` is used for grading: you may change its value in the cell where it is defined, but do not set it anywhere else in the notebook.
3. Keep `ex4_helpers.py` in the same folder as this notebook: the tests use it. You do not need to read it.
4. Do not delete cells. The cells that say "Do not delete this cell" hold tests that run after you submit.

**Deliverables**:

Submit the completed notebook (`ex4.ipynb`) and your trained model (`best_model.pth`), without renaming them. You do not need to submit `alice.txt` or `ex4_helpers.py`.

In [1]:
skip_training = False   # You can set it to True if you want to run inference on your trained model.

In [ ]:
# Do not delete this cell

In [ ]:
# Do not delete this cell

### Import the necessary libraries

In [2]:
import os
import random
import re

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

try:
    from ex4_helpers import start_grading
except ImportError:
    raise ImportError('ex4_helpers.py was not found: keep it in the same folder as this notebook.') from None
AUTOGRADING = start_grading(globals().get('autograding', False))    # True only while the autograder grades the notebook

In [3]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


### 1. Load and Preprocess the Text Dataset

We will be using *Alice's Adventures in Wonderland* by Lewis Carroll as our dataset. You can download it from [Project Gutenberg](https://www.gutenberg.org/):

[Alice's Adventures in Wonderland by Lewis Carroll (Project Gutenberg Page)](https://www.gutenberg.org/ebooks/11) \
[Direct Text File Download](https://www.gutenberg.org/files/11/11-0.txt)

We’ve chosen Alice's Adventures in Wonderland as a relatively small text to make training still manageable on a CPU. However, you are highly encouraged to explore other texts from Project Gutenberg or other public domain sources. Submit a model trained on Alice, though: grading uses this text.

This section contains the following steps:
1. Load the dataset into Python
2. Remove metadata to focus on the main part of the text
3. Clean the text by removing special characters and converting it to lowercase
   
The goal is to preprocess the dataset by filtering out any metadata that is not part of the text, converting the text to lowercase, and removing unnecessary punctuation. We will also build a dictionary to map each unique character to a unique integer.

#### 1.1. Load the Dataset

We start by loading the text dataset into Python. The dataset should be a plain text file. The first step is to load and inspect a small portion of the raw text to understand its structure to identify any unwanted metadata or special characters that should be removed during preprocessing.

Save the text file as `alice.txt` in the same folder as this notebook, or set `txt_path` below to the place where you saved it. During grading, the autograder reads its own copy of the same text.

In [4]:
txt_path = 'alice.txt' # replace 'alice.txt' with your txt path

In [6]:
# Reads the text. During grading, the autograder reads its own copy of the same text.
from ex4_helpers import text_path

with open(text_path(globals().get('txt_path', 'alice.txt'), globals().get('TA_txt_path'), AUTOGRADING), 'r', encoding='utf-8') as file:
    raw_text = file.read()

print('===First 1500 characters before any processing:\n\n')
print(raw_text[:1500])

print('\n\n\n===Ending characters before any processing:\n')
print(raw_text[-19000:-17000])

===First 1500 characters before any processing:


*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence




CHAPTER I.
Down the Rabbit-Hole


Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures

#### 1.2. Remove Metadata and Focus on the Main Text

Text files may contain introductary or ending metadata such as copyright information. We want to focus only on the main body of the text. For Alice's Adventures in Wonderland, we remove everything before the first chapter and after the Project Gutenberg closing markers.

In [7]:
# For this example, we are removing everything before 'CHAPTER I.\nDown the Rabbit-Hole'
# and after the end marker
start_index = raw_text.find('CHAPTER I.\nDown the Rabbit-Hole')

end_index = raw_text.find('*** END OF THE PROJECT GUTENBERG') # closing markers of Project Gutenberg

trimmed_text = raw_text[start_index:end_index]

print('===Text after removing metadata:\n')
print(trimmed_text[:1500])

===Text after removing metadata:

CHAPTER I.
Down the Rabbit-Hole


Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”

So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy and stupid), whether the pleasure of
making a daisy-chain would be worth the trouble of getting up and
picking the daisies, when suddenly a White Rabbit with pink eyes ran
close by her.

There was nothing so _very_ remarkable in that; nor did Alice think it
so _very_ much out of the way to hear the Rabbit say to itself, “Oh
dear! Oh dear! I shall be late!” (when she thought it over afterwards,
it occurred to her that she ought to have wondered at this, but at the
time it all seemed quite natural); but when the Rabbit a

#### 1.3. Clean the Text

Next, we preprocess the text by removing any special characters, leaving only alphanumeric characters, and normalizing spaces. We also convert all text to lowercase to standardize the format. This helps the model learn without case sensitivity or irrelevant symbols.

##### Steps to follow:
##### 1. Convert text to lowercase:
First, convert the text to lowercase to avoid treating uppercase and lowercase letters as different characters.
##### 2. Remove special characters:
Then, you need to remove any character that is not a letter `(a-z)`, a number `(0-9)`, or a space `\s`.
##### 3. Handling double spaces:
After removing characters, there may be extra spaces in the text. Make sure that sequences of multiple spaces are reduced to just a single space.

**Hint:** You can use the `re.sub()` method of regular expressions library `re` to replace the patterns. In this case, you will be replacing non-alphanumeric characters (`[^a-z0-9\s]`) with spaces, and whitespace sequences (`\s+`) with a single space.

In [9]:
def preprocess_text(text):
    """
    Preprocesses the input text by i. converting it to lowercase,
    ii. removing non-alphanumeric characters (except spaces),
    iii. and normalizing spaces.

    Args:
    text -- The raw input text as a string

    Returns:
    cleaned_text -- The processed text where all the preprocessing steps are applied
    """
    # 1. Convert text to lowercase
    cleaned_text = text.lower()

    # 2. Remove special characters
    cleaned_text = re.sub(r'[^a-z0-9\s]', '', cleaned_text)

    # 3. Remove double spaces
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)    

    return cleaned_text.strip()

cleaned_text = preprocess_text(trimmed_text)
print('Text after cleaning and converting to lowercase:\n')
print(cleaned_text[:1000])


Text after cleaning and converting to lowercase:

chapter i down the rabbithole alice was beginning to get very tired of sitting by her sister on the bank and of having nothing to do once or twice she had peeped into the book her sister was reading but it had no pictures or conversations in it and what is the use of a book thought alice without pictures or conversations so she was considering in her own mind as well as she could for the hot day made her feel very sleepy and stupid whether the pleasure of making a daisychain would be worth the trouble of getting up and picking the daisies when suddenly a white rabbit with pink eyes ran close by her there was nothing so very remarkable in that nor did alice think it so very much out of the way to hear the rabbit say to itself oh dear oh dear i shall be late when she thought it over afterwards it occurred to her that she ought to have wondered at this but at the time it all seemed quite natural but when the rabbit actually took a watch ou

In [10]:
# Visible test of preprocess_text(). Task 1 has no points, but the later tasks use the cleaned text.
from ex4_helpers import run_test, check_text_cleaning
run_test('Task 1: text cleaning (visible test, 0 points)', lambda: check_text_cleaning(raw_text, cleaned_text))

Task 1: text cleaning (visible test, 0 points): passed


### 2. Character-Level Encoding (2 points)

In this step, we convert the cleaned text into a format that the model can understand. Since we are working with character-level encoding, each individual character will be treated as a token. This allows the LSTM to learn patterns at the character level and generate text one character at a time.



#### 2.1 Character-Level Vocabulary

In this section, you will create a vocabulary of from the cleaned text and map each element to a unique integer.

##### Steps to follow:
##### 1. Create a vocabulary of unique characters:
First, you need to extract all the unique characters from the cleaned text. You can use Python's built-in `set()` function to find unique characters. For consistency and easier debugging, you should also sort the unique characters.
##### 2. Construct mapping between characters and integers:
Once you have the unique characters, create a dictionary `char_to_int` that maps each character to a unique integer to represent each character during training. You should also create a reverse mapping `int_to_char` that maps integers back to characters to be used when decoding the text later.


In [15]:
def create_char_mappings(cleaned_text):
    """
    Creates character-to-integer and integer-to-character mappings from the cleaned text.

    Args:
    cleaned_text -- The cleaned input text as a string

    Returns:
    char_to_int -- A dictionary mapping each unique character to an integer
    int_to_char -- A dictionary mapping each integer back to its corresponding character
    """
    char_to_int = {}
    int_to_char = {}
    sorted_set = sorted(set(cleaned_text))
    for i in range(len(sorted_set)):
        char_to_int[sorted_set[i]] = i
        int_to_char[i] = sorted_set[i]

    return char_to_int, int_to_char

char_to_int, int_to_char = create_char_mappings(cleaned_text)
print('Character to Integer Mapping:')
for char, idx in list(char_to_int.items()):
    print(f"'{char}' : {idx}")

Character to Integer Mapping:
' ' : 0
'a' : 1
'b' : 2
'c' : 3
'd' : 4
'e' : 5
'f' : 6
'g' : 7
'h' : 8
'i' : 9
'j' : 10
'k' : 11
'l' : 12
'm' : 13
'n' : 14
'o' : 15
'p' : 16
'q' : 17
'r' : 18
's' : 19
't' : 20
'u' : 21
'v' : 22
'w' : 23
'x' : 24
'y' : 25
'z' : 26


#### 2.2 Encode the Text into Integers

During training, the model will use the encoded representation of the cleaned text as the input. In this section, you need to convert each character in the cleaned text to its corresponding integer using `char_to_int` dictionary.

In [19]:
def encode_text(cleaned_text, char_to_int):
    """
    Encodes the cleaned text into an array of integers.

    Args:
    cleaned_text -- The cleaned input text as a string
    char_to_int -- Characters to integer mapping

    Returns:
    encoded_chars -- Numpy array of integers representing the encoded characters from the text
    """
    encoded_chars = []
    for char in cleaned_text:
        encoded_chars.append(char_to_int[char])
    return encoded_chars

encoded_chars = encode_text(cleaned_text, char_to_int)
print(f"Total encoded characters: {len(encoded_chars)}")
print('First 100 encoded characters:')
print(encoded_chars[:100])

Total encoded characters: 134046
First 100 encoded characters:
[3, 8, 1, 16, 20, 5, 18, 0, 9, 0, 4, 15, 23, 14, 0, 20, 8, 5, 0, 18, 1, 2, 2, 9, 20, 8, 15, 12, 5, 0, 1, 12, 9, 3, 5, 0, 23, 1, 19, 0, 2, 5, 7, 9, 14, 14, 9, 14, 7, 0, 20, 15, 0, 7, 5, 20, 0, 22, 5, 18, 25, 0, 20, 9, 18, 5, 4, 0, 15, 6, 0, 19, 9, 20, 20, 9, 14, 7, 0, 2, 25, 0, 8, 5, 18, 0, 19, 9, 19, 20, 5, 18, 0, 15, 14, 0, 20, 8, 5, 0]


In [21]:
# Visible test of the character encoding
from ex4_helpers import run_test, check_encoding
run_test('Task 2: character encoding (visible test)',
         lambda: check_encoding(cleaned_text, encoded_chars, char_to_int, int_to_char))

Task 2: character encoding (visible test): passed


In [ ]:
# Do not delete this cell. It contains hidden tests that are run after submission.

### 3. Batch Generation for Training (6 points)
In this step, you will implement the function `get_batches()` that splits the encoded data into smaller batches for training. Each batch will have input sequences `x` and target sequences `y` where `y` is `x` shifted by one position.  This means that the model is trained to generate the next character in the sequence based on the previous ones.

##### Steps to follow:
##### 1. Handle step_size:
The step size determines how much the window moves across the data after each sequence is generated. If `step_size` is not provided, it is set to `seq_length`. This means the sequences will not overlap. A smaller `step_size` allows for overlapping sequences.

##### 2. Calculate the number of batches:
When calculating how many batches you can generate from the input data, there are some key factors to consider:
1. Sequence Length: Each input sequence in a batch will contain a specific number of tokens (representing the characters). The longer your sequence length is, the fewer total sequences you can generate from the input data.
2. Step Size: A smaller step size results in more overlap between sequences and it allows you to generate more sequences from the same input. If the step size is larger, there will be less overlap (or none at all if step size equals sequence length), leading to fewer sequences in total.
3. Batch Size: Once you generate sequences, you need to group them into batches for efficient training. The batch size defines how many sequences are grouped in each batch. A larger batch size means fewer batches because more sequences are grouped together in each batch.

Make sure to generate full number of batches.

##### 3. Trim the input array:
If the input data does not perfectly divide into batches, trim the array so it contains only full batches. Avoid having incomplete sequences at the end.

##### 4. Generate batches:
Use nested loops to generate batches:
- `x` will be the input sequence of length `seq_length`.
- `y` will be target sequence, which is `x` shifted by one position (token).

##### 5. Store and return batches:
Store the input and target sequences in separate arrays (`x_batches` and `y_batches`) and return them as NumPy arrays to be used in training.

**Expected Shape**:
- Each batch in `x_batches` and `y_batches` should have the shape `(batch_size, seq_length)`.
- The returned `x_batches` and `y_batches` should be NumPy arrays with shapes `(num_batches, batch_size, seq_length)`.

**Important Notes**:
- Support for both overlapping and non-overlapping sequences using `step_size`.
- Handle edge cases where the data does not fit perfectly into full batches.
- Think about how you are generating both the `x` and `y` sequences. Their size should match but `y` should always be one token ahead of `x`.



In [27]:
def get_batches(arr, batch_size, seq_length, step_size=None):
    """
    Generates batches of input and target sequences from the given array.

    Args:
        arr (array-like): Encoded text as an array of integers.
        batch_size (int): Number of sequences per batch.
        seq_length (int): Number of characters in each sequence.
        step_size (int, optional): Steps to move the window for the next sequence.

    Returns:
        tuple (x_batches, y_batches): A tuple of numpy arrays of input and target sequences,
                                      each has shape (num_batches, batch_size, seq_length)
    """
    if step_size is None:
        step_size = seq_length
    x_sequences = []
    y_sequences = []
    # YOUR CODE HERE
    for i in range(0, len(arr) - seq_length, step_size):
        x_sequences.append(arr[i:i + seq_length])
        y_sequences.append(arr[i + 1:i + seq_length + 1])

        # Convert to numpy arrays
    x_sequences = np.array(x_sequences)
    y_sequences = np.array(y_sequences)

    # Keep only complete batches
    num_sequences = (len(x_sequences) // batch_size) * batch_size

    x_sequences = x_sequences[:num_sequences]
    y_sequences = y_sequences[:num_sequences]

    # Reshape into batches
    x_batches = x_sequences.reshape(-1, batch_size, seq_length)
    y_batches = y_sequences.reshape(-1, batch_size, seq_length)


    return x_batches, y_batches

In [28]:
# Visible test of get_batches() without overlap (step_size = seq_length), on an array as long as the cleaned text
from ex4_helpers import run_test, check_batch_shapes
run_test('Task 3: get_batches() without overlap (visible test)', lambda: check_batch_shapes(get_batches))

Task 3: get_batches() without overlap (visible test): passed


In [29]:
# Visible test of get_batches() with overlap (step_size = 50, seq_length = 100)
from ex4_helpers import run_test, check_batch_shapes
run_test('Task 3: get_batches() with overlap (visible test)', lambda: check_batch_shapes(get_batches, step_size=50))

Task 3: get_batches() with overlap (visible test): passed


In [30]:
# Display for y shift and  step_size
def display_batch_generation(arr, char_to_int, int_to_char):
    batch_size, seq_length, step_size = 8, 10, 5  # Setting step_size for overlap between sequences

    x_batches, y_batches = get_batches(arr, batch_size, seq_length, step_size)

    # Display batch number 10
    x_chars = ''.join([int_to_char[idx] for idx in x_batches[10][0]])
    y_chars = ''.join([int_to_char[idx] for idx in y_batches[10][0]])

    print('='*50)
    print('Displaying a Single Batch')
    print('='*50)
    for i in range(batch_size):
        x_chars = ''.join([int_to_char[idx] for idx in x_batches[10][i]])
        y_chars = ''.join([int_to_char[idx] for idx in y_batches[10][i]])

        print(f"[{x_chars}]  -->  [{y_chars}]")
    print('='*50)
display_batch_generation(encoded_chars, char_to_int, int_to_char )

Displaying a Single Batch
[ade her fe]  -->  [de her fee]
[er feel ve]  -->  [r feel ver]
[el very sl]  -->  [l very sle]
[ry sleepy ]  -->  [y sleepy a]
[eepy and s]  -->  [epy and st]
[and stupid]  -->  [nd stupid ]
[tupid whet]  -->  [upid wheth]
[ whether t]  -->  [whether th]


In [ ]:
# Do not delete this cell. It contains hidden tests that are run after submission.

### 4. Define the Character-Level LSTM Model (6 points)
In this step, you will implement the CharLSTM class, which processes sequences of characters and predicts the next character in the sequence. The model will learn sequential patterns in the data and store information over time using hidden states.

##### Key Components:
##### 1. Single Multi-Layer LSTM (see [nn.LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)):
- Use a single `nn.LSTM` module configured as a multi-layer LSTM by setting the `num_layers` parameter to specify the number of stacked layers within this LSTM.
- Dropout set in `nn.LSTM` automatically applies dropout between the internal LSTM layers (e.g., between the 1st and 2nd layers if num_layers=2). This dropout is only applied between the internal LSTM layers and does not affect the final output layer.

##### 2. Dropout Layer (see [nn.Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html)):
- Define an additional dropout layer to be applied after the final LSTM layer. This helps to prevent overfitting.

##### 3. Fully Connected Layer (see [nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)):
- After the LSTM layers, a fully connected layer maps the hidden states to process them as a probability distribution over the vocabulary to predict the next character in the sequence.

##### 4. Hidden State Initialization:
- The LSTM's hidden and cell states will be initialized with zero values before each batch is processed and will be updated as the model processes the sequence.

##### Steps to follow:
- You need to implement the following methods in the `CharLSTM` class:
   - `__init__()` to define the architecture.
   - `forward()` to handle the forward pass.
   - `init_hidden()` to initialize the hidden and cell states before each batch.
- The model architecture should follow this flow:
    - One `nn.LSTM`: Define a single multilayer LSTM with `num_layers=num_layers`, output dimensionality `hidden_dim`, and internal dropout of `dropout_prob` between stacked layers.
    - Dropout after LSTM: Apply a separate dropout layer `nn.Dropout` with the probability of `dropout_prob` to the LSTM’s output after all layers.
    - Fully Connected Layer: Define a fully connected layer to output logits for each character in the sequence.

**Important Notes**:
- Ensure that you use the provided parameters (e.g., num_layers, hidden_dim) when defining the model architecture. Avoid hardcoding values (like `num_layers=2`)
- One-hot encoding will be applied in the training loop before the data is passed to the model. This will adjust the shape of x for each batch to `(batch_size, seq_length, input_dim)`, where input_dim is the vocabulary size.  You do not need to handle this encoding within the CharLSTM class.
- We will use cross-entropy loss as the loss function. The cross-entropy loss combines the softmax operation and the negative log-likelihood loss in a single step. The loss function takes the raw outputs (logits) from the fully connected layer and internally converts them to a probability distribution. Therefore, you do not need to apply Softmax separately.
- Ensure the hidden states are initialized on the same device as the model parameters to avoid device mismatch errors.
- Make sure to configure the LSTM as one `nn.LSTM` module with `num_layers` rather than separate layers. This approach is critical for testing.
- Do not confuse stacked LSTM (configured with `num_layers`) with bidirectional LSTM.
- Carefully check LSTM parameters and make sure that input and output shapes are correct. Pay special attention to the difference between batched and unbatched input shapes.
- Consider how `batch_first` parameter of `nn.LSTM` aligns with the shape of your data after one-hot encoding and think about how the batch dimension should be treated within the LSTM.

In [47]:
class CharLSTM(nn.Module):
    """
    Character-Level Multi-Layer LSTM Model

    This model processes sequences of characters and predicts the next character in the sequence.
    """

    def __init__(self, num_layers, input_dim, hidden_dim, output_dim, dropout_prob):
        """
        Initializes the CharLSTM model with the specified parameters.

        Args:
            num_layers (int): Number of LSTM layers
            input_dim (int): Dimensionality of the input (e.g. one-hot encoded input size)
            hidden_dim (int): Dimensionality of the LSTM hidden layer.
            output_dim (int): Dimensionality of the output.
            dropout_prob (float): Dropout after each layer.
        """
        super(CharLSTM, self).__init__()
        # Save hidden dimension and number of layers for hidden state initialization
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Define a single LSTM module configured as a multi-layer LSTM
        # Internal dropout of dropout_prob is applied between layers
        # Apply additional dropout after LSTM
        # Apply dense layer at the end
        # YOUR CODE HERE
        self.lstm = nn.LSTM(num_layers=num_layers, 
                              input_size=input_dim, 
                              hidden_size=hidden_dim, 
                              dropout=dropout_prob,
                              batch_first=True
                              )
        self.dropout = nn.Dropout(p=dropout_prob)
        self.linear = nn.Linear(hidden_dim, output_dim)


    def forward(self, x, hidden):
        """
        Performs the forward pass of the CharLSTM model.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, seq_length, input_dim).
            hidden (tuple): Tuple of (h0, c0), where each is a tensor of shape (num_layers, batch_size, hidden_dim).

        Returns:
            out (torch.Tensor): Output tensor of shape (batch_size, seq_length, output_dim).
            (h, c) (tuple): Updated hidden states (h, c) for each LSTM layer.
                - h (torch.Tensor): Final hidden state
                - c (torch.Tensor): Final cell state
        """

        # Pass the input through the LSTM
        # The LSTM output 'out' has shape (batch_size, seq_length, hidden_dim)
        # 'h' and 'c' represent the final hidden and cell states for each layer
        # YOUR CODE HERE
        out, (h,c) = self.lstm(x, hidden)
        out = self.dropout(out)
        out = self.linear(out)

        # Return the final output and the updated hidden states
        return out, (h, c)

    def init_hidden(self, batch_size):
        """
        Initializes the hidden and cell states to zeros for each LSTM layer.

        Args:
            batch_size (int): The batch size for the current data.

        Returns:
            (h0, c0) (tuple): Tuple of initial hidden states (h0, c0) for each LSTM layer.
            - h0 (torch.Tensor): Initial hidden state
            - c0 (torch.Tensor): Initial cell state
        """
        # Set the device to match the model's device to prevent device mismatch errors
        device = next(self.parameters()).device

        # Initialize hidden state (h0) and cell state (c0) to zeros
        # Check the expected shape of hidden states in pytorch nn.LSTM documentation
        # YOUR CODE HERE
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)

        return (h0, c0)

In [45]:
# Visible test of the dropout in CharLSTM
from ex4_helpers import run_test, check_dropout
run_test('Task 4: dropout (visible test)', lambda: check_dropout(CharLSTM, device))

Task 4: dropout (visible test): passed


In [46]:
# Visible test of the shapes of the output and the hidden states of CharLSTM
from ex4_helpers import run_test, check_lstm_shapes
run_test('Task 4: shapes of the output and the hidden state (visible test)', lambda: check_lstm_shapes(CharLSTM, device))

Task 4: shapes of the output and the hidden state (visible test): passed


In [ ]:
# Do not delete this cell. It contains hidden tests that are run after submission.

#### 5. Train the Model (3 points)

In this task, you will implement the training loop for the CharLSTM model.

In the train() function you are given:
- Optimizer and loss function: The [Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) optimizer and [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) are used. Do not change the optimizer and loss function.
- Preparing the Batches: The `get_batches()` function is used to divide the encoded text data into smaller input (x) and target (y) sequences.

##### Steps to Follow:
Outside of the iteration, initialize the hidden states using the `init_hidden()` method.

For each iteration:
##### 1. Encoding the input sequence:
- For each batch, convert the input to one-hot representations (see [F.one_hot](https://pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html))

##### 2. Detach Hidden States:
- Detach the hidden states after each batch to avoid backpropagating through previous batches to ensure efficient training.

##### 3. Forward Pass and Loss Calculation:
- For each batch, perform forward pass by passing the input x to the model.
- The model will output logits, which will represent the predicted probabilities for the next character in the sequence when passed to the cross-entropy loss.
- Use cross-entropy loss to compare the predicted output to the target y and calculate the error for the current batch.

##### 4. Backpropagation and Parameter Update:
- After calculating the loss compute the gradients with backward pass.
- Update the model parameters using the optimizer.


**Hints**:
- Do not forget to zero the gradients.
- Ensure that the logits and target tensors are reshaped appropriately to match the expected size for the loss function specified  [here](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html). \
Logits should have shape (batch_size * seq_length, vocab_size) and targets should have shape (batch_size * seq_length).

In [58]:
def train(model, encoded_chars, vocab_size, num_epochs, batch_size,
          seq_length, step_size, learning_rate, save_path=None, verbose=True):
    """
    Train the CharLSTM model on encoded text data.

    Arguments:
    model -- The LSTM model
    encoded_chars -- Encoded data (characters)
    vocab_size -- Size of the vocabulary
    num_epochs -- Number of training epochs
    batch_size -- Batch size for training
    seq_length -- Sequence length for each batch
    learning_rate -- Learning rate for the optimizer
    save_path -- Path to save the trained model (optional)
    """

    model.train()  # Set model to training mode
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)  # Initialize Adam optimizer
    criterion = nn.CrossEntropyLoss()  # Cross entropy loss function

    # Prepare batches
    x_batches, y_batches = get_batches(encoded_chars, batch_size, seq_length, step_size)
    num_batches = len(x_batches)

    for epoch in range(num_epochs):
        total_loss = 0
        # Progress bar for the current epoch
        batch_loader = tqdm(zip(x_batches, y_batches), total=num_batches,
                            leave=True, desc=f'Epoch {epoch+1}/{num_epochs}')

        # Initialize hidden states for both LSTM layers
        # YOUR CODE HERE
        hidden = model.init_hidden(batch_size)

        for x, y in batch_loader:
            x = torch.as_tensor(x, dtype=torch.long).to(device)
            y = torch.as_tensor(y, dtype=torch.long).to(device) # target

            # One-hot encoding
            x = F.one_hot(x, num_classes=vocab_size).float()
            
            # Make sure to have consistent variable naming with the rest of the code (e.g., loss)
            optimizer.zero_grad()
            output, hidden = model(x, hidden)
            hidden = tuple(h.detach() for h in hidden)
            
            # Loss
            output = output.reshape(-1, vocab_size)
            y = y.reshape(-1)
            loss = criterion(output, y)

            # Backward
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Print the average loss for the current epoch
        if verbose:
            print(f'Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / num_batches:.4f}')

        # Optional model saving
        # Let us save it each epoch since training takes a while and you want to stop in the middle
        if save_path:
            torch.save(model.state_dict(), save_path)
            print(f'Your trained model at epoch {epoch} is saved successfully!')

    return total_loss / num_batches

In [59]:
# Visible checks of train(), with a small model and random batches: the model gets one-hot input and hidden states
# that are detached from the previous batch. They have no points, but help you find mistakes.
from ex4_helpers import run_test, check_train_basics
run_test('Task 5: one-hot input and detached hidden states (visible check, 0 points)',
         lambda: check_train_basics(train, CharLSTM, device))

Task 5: one-hot input and detached hidden states (visible check, 0 points): passed


In [60]:
# Visible test of train(), with a small model and random batches: what train() gives the loss function
from ex4_helpers import run_test, check_train_loss
run_test('Task 5: what train() gives the loss function (visible test)', lambda: check_train_loss(train, CharLSTM, device))

Task 5: what train() gives the loss function (visible test): passed


#### Model Initialization

The model settings in the next cell are fixed: two LSTM layers (`num_layers=2`), a hidden dimension of 400 and dropout of 0.1. Grading builds `CharLSTM` with these settings and loads your `best_model.pth` into it, so do not change them. (To try other settings, use cells of your own, and submit a model with the settings below.) You can change the training settings in the cell after it: `num_epochs`, `batch_size`, `seq_length`, `step_size` and `learning_rate`.

**How your model is graded.** After you submit, a hidden test computes the average cross-entropy loss of your saved `best_model.pth` over the whole cleaned text, in evaluation mode (without dropout), in batches of 50 sequences of 100 characters. It must be below 1.05. This loss is lower than the training loss printed during training, which is measured with dropout: a final training loss below 0.99 is enough. With the settings below, 50 epochs are enough; on a laptop CPU they take about 10 minutes, and the model is saved after every epoch.

In [88]:
hidden_dim = 400
dropout_prob=0.2
num_layers=3
vocab_size = len(char_to_int)
model = CharLSTM(num_layers, vocab_size, hidden_dim, vocab_size, dropout_prob)
model = model.to(device)
print(model)

CharLSTM(
  (lstm): LSTM(27, 400, num_layers=3, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
  (linear): Linear(in_features=400, out_features=27, bias=True)
)


In [89]:
num_epochs = 100 # Train for *at least* 50 epochs to meet the min loss of 0.99
batch_size = 50
seq_length=100
step_size=100
learning_rate=0.001


In [90]:
from ex4_helpers import training_allowed

if training_allowed(skip_training):    # never during grading: the autograder grades the best_model.pth you submitted
    loss = train(
        model=model,
        encoded_chars=encoded_chars,
        vocab_size=vocab_size,
        num_epochs=num_epochs,
        batch_size=batch_size,
        seq_length=seq_length,
        step_size=step_size,
        learning_rate=learning_rate,
        save_path='best_model.pth'
    )
else:
    model.load_state_dict(torch.load('best_model.pth', weights_only=False, map_location=device))
    print('Loaded weights from your saved model successfully!')

Epoch 1/100:   0%|          | 0/26 [00:00<?, ?it/s]

Epoch 1/100: 100%|██████████| 26/26 [00:01<00:00, 17.88it/s]


Epoch 1/100, Loss: 2.8955
Your trained model at epoch 0 is saved successfully!


Epoch 2/100: 100%|██████████| 26/26 [00:01<00:00, 23.08it/s]


Epoch 2/100, Loss: 2.8197
Your trained model at epoch 1 is saved successfully!


Epoch 3/100: 100%|██████████| 26/26 [00:01<00:00, 22.54it/s]


... (191 outputs left out)

Epoch 99/100: 100%|██████████| 26/26 [00:01<00:00, 23.65it/s]


Epoch 99/100, Loss: 0.3651
Your trained model at epoch 98 is saved successfully!


Epoch 100/100: 100%|██████████| 26/26 [00:01<00:00, 21.57it/s]

Epoch 100/100, Loss: 0.3655
Your trained model at epoch 99 is saved successfully!


In [64]:
# Do not delete this cell. It contains hidden tests that are run after submission.

### 6. Text Generation (3 points)
The `text_generation()` function will use a trained model to create new text based on a given starting string.

In this process, the model becomes autoregressive by using its own predictions as inputs for the next steps. Starting with an initial string, the model produces characters one by one, feeding each newly generated character back into itself as input.


1. Input start string:
The function begins with a starting string, which is converted into a sequence of integers using the character-to-integer mapping.
2. Generate text:
The model will take this sequence as input, predict the next character, and add it to the text. This process is repeated for a specified number of characters `predict_len`.
3. Output probabilities

There are a few steps in the function for you to implement.

##### Steps to follow:
1. One-Hot encoding: Apply one-hot encoding to the input sequence as you did in the training loop.

2. Forward pass: Feed the one-hot encoded input through the model to obtain the output logits and the updated hidden state.

3. Extract the last output: You only need the output from the **last time step** to predict the next character. Slice the output and get the last element along the sequence dimension. If you did the training well, the generated text should mostly include meaningful words.

4. Temperature scaling: To control the randomness in prediction, divide the output logits by the temperature parameter. The temperature should be in range (0,1]. Higher temperatures produce more random text, while lower temperatures produce more predictable results. You can observe the variations in the generated text by experimenting with different temperature values.

You can try generating text after training for just one epoch to observe the model's initial behavior. Depending on your temperature setting, the generated text might be a repetition of a single character or random sequences.

In [91]:
def generate_text(model, start_str, char_to_int, int_to_char, vocab_size, predict_len=100, temperature=1.0):
    """
    Generate text using the trained model.

    Arguments:
    model -- Trained RNN model
    start_str -- String to start generating from
    char_to_int -- Dictionary mapping characters to integers
    int_to_char -- Dictionary mapping integers back to characters
    vocab_size -- Size of the vocabulary
    predict_len -- Number of characters to generate
    temperature -- Float controlling randomness in predictions (higher is more random)

    Returns:
    generated_text -- The generated text as a string
    """
    model.eval()  # Set model to evaluation mode

    # Encode the starting string
    input_seq = [char_to_int[char] for char in start_str]
    input_seq = torch.tensor(input_seq).long().to(device).unsqueeze(0)  # Add batch dimension

    hidden = model.init_hidden(1)  # Batch size of 1 for generating text

    generated_text = start_str

    with torch.no_grad():  # inference
        # One-hot encoding
        x = F.one_hot(input_seq, num_classes=vocab_size).float()
        output, hidden = model(x, hidden)

        for _ in range(predict_len):

            # Get logits
            logits = output[:, -1, :] / temperature

            # Convert output to probabilities using softmax
            probabilities = F.softmax(logits, dim=-1).detach().cpu().numpy()

            # Randomly sample based on the output probabilities
            next_char_index = np.random.choice(range(vocab_size), p=probabilities.ravel())

            # Add the predicted character to the generated text
            next_char = int_to_char[next_char_index]
            generated_text += next_char

            # Update the input sequence - shift left to preserve the input length and add the new character
            input_seq = torch.cat([input_seq[:, 1:], torch.tensor([[next_char_index]]).to(device)], dim=1)

            # Re-apply one-hot encoding for the next iteration
            x = F.one_hot(input_seq, num_classes=vocab_size).float()
            output, hidden = model(x, hidden)


    return generated_text

In [92]:
start_str = 'we re all '
predict_len = 1000
temperature = 0.5
generated_text = generate_text(model,
                               start_str,
                               char_to_int,
                               int_to_char,
                               vocab_size,
                               predict_len=predict_len,
                               temperature=temperature)
print(generated_text)

we re all drighteded in a moment why she thought alice did not go on as the cats growl to large i drink yese i wonder what sort of pray she felt that was little besidess and those words out of the hall wasting before alice was the game the queen and more for the mock turtle but a box of course when they all drrized and finish this she was reading over and the white rabbit with a conversation alice venture to say into the hatter questions was in the world go and great surpoce in the shall i dont know its an arm very will you wont you join the king said the hatters the march hare with great curious creatures you did not came but very well to the great quite run by the hall wasting i should like heard out of the hall happens when and the queen stand of the grave cried the gryphon and she went on the baby excopt of you contenth she heard while the march hare with dinah i cant go no sort of might be a little way of seeped to come in seeds of kind and the march hall when id a good dear or ol

In [ ]:
# Do not delete this cell. It contains hidden tests that are run after submission.

In [ ]:
# Do not delete this cell. It contains hidden tests that are run after submission.

In [ ]:
# Do not delete this cell. It contains hidden tests that are run after submission.

In [ ]:
# Do not delete this cell

**Closing remarks**:

1. Consider experimenting with more complex architectures by adding additional LSTM layers or increasing the hidden dimension size. Keep in mind that even with GPU resources this can take a while.

2. In this task, we used one-hot encoding to represent inputs. However, you can experiment with the `nn.Embedding` module in PyTorch, which creates better representations for input characters and may improve model performance.

3. For more complex models, you do not need to remove special characters, like punctuation and new lines, during preprocessing. Keeping these characters is helpful especially if you want to generate text in different styles, such as Shakespearean sonnets, where line breaks and punctuation are important for preserving the text style.
